# COU 07506 — Week 3: Lab 1B — Convolution, Correlation and Difference Equations

**Name:** WILLIAM KEZA OSCAR  
**Registration number:** 240242431351  
**Date:** NOV 25, 2026  
**Group:** GROUP 13

**Git repository:** https://github.com/keza-oscar/COU07506-labs

Submitted file: `COU07506_Week03_240242431351.ipynb`.


## Pre-lab

**(a)** With $x=\{1,-1,2\}$, $h=\{2,1,0,1\}$, linear convolution has length $3+4-1=6$. Term by term: $y[0]=2$, $y[1]=1-2=-1$, $y[2]=0-1+4=3$, $y[3]=1+0+2=3$, $y[4]=-1+0=-1$, $y[5]=2$.  
**Answer:** $\{2,-1,3,3,-1,2\}$.

**(b)** $y(n)=n\,x(n)$ is **not time-invariant**. Delay the input by 5: $T\{x[n-5]\}=n\,x[n-5]$, but delaying the output gives $y[n-5]=(n-5)x[n-5]$. These differ because the multiplier depends explicitly on $n$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from scipy.io import wavfile
from IPython.display import Audio
plt.rcParams['figure.figsize'] = (9, 3.5)

## Task 3.1 — Convolution from the definition

In [ ]:
import time

def conv(x, h):
    N = len(x) + len(h) - 1
    y = np.zeros(N)
    for n in range(N):
        for k in range(len(x)):
            if 0 <= n - k < len(h):
                y[n] += x[k] * h[n - k]
    return y

print('test:', conv([1, 2, 3, 1], [1, 1, 1]))
print('numpy:', np.convolve([1, 2, 3, 1], [1, 1, 1]))

x0, h0 = np.array([1, -1, 2]), np.array([2, 1, 0, 1])
print('pre-lab (own):', conv(x0, h0))
print('pre-lab (numpy):', np.convolve(x0, h0))

rng = np.random.default_rng(0)
xa, ha = rng.standard_normal(2000), rng.standard_normal(2000)
t0 = time.perf_counter(); conv(xa, ha); t_loop = time.perf_counter() - t0
t0 = time.perf_counter(); np.convolve(xa, ha); t_np = time.perf_counter() - t0
print(f'own loop: {t_loop:.4f} s, np.convolve: {t_np:.6f} s, '
      f'speed-up: {t_loop/t_np:.0f}x')

## Task 3.2 — Smoothing a noisy signal by convolution

In [ ]:
rng = np.random.default_rng(3)
n = np.arange(300)
xn = np.sin(2*np.pi*n/50) + 0.5*rng.standard_normal(300)
plt.figure(figsize=(9, 3.5))
plt.plot(n, xn, label='noisy x(n)')
for M in [3, 9, 25]:
    h = np.ones(M)/M
    ys = np.convolve(xn, h, mode='same')
    plt.plot(n, ys, label=f'moving average M={M}')
plt.title('Task 3.2: smoothing with moving-average filters')
plt.xlabel('n (samples)'); plt.ylabel('amplitude'); plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

**Trade-off:** larger $M$ reduces random noise more (averaging suppresses zero-mean noise by about $\sqrt{M}$) but also attenuates and delays the wanted 50-sample sine wave more, blurring peaks — a low-pass filter with wider main lobe and more delay.

## Task 3.3 — Testing linearity and time-invariance numerically

In [ ]:
rng = np.random.default_rng(0)
x1, x2 = rng.standard_normal(50), rng.standard_normal(50)
a, b = 2.0, -3.0
systems = {
    'x^2': lambda x: x**2,
    'n x(n)': lambda x: np.arange(len(x)) * x,
    'moving average': lambda x: np.convolve(x, np.ones(3)/3)[:len(x)],
}
for name, T in systems.items():
    linear = np.allclose(T(a*x1 + b*x2), a*T(x1) + b*T(x2))
    xd = np.concatenate([np.zeros(5), x1])[:50]
    yd = np.concatenate([np.zeros(5), T(x1)])[:50]
    tinv = np.allclose(T(xd), yd)
    print(f'{name:15s} linear={linear!s:5s} time-invariant={tinv}')

**Explanation:** $x^2$ is nonlinear (squaring does not preserve superposition) and time-invariant (the rule has no explicit $n$). $n\,x(n)$ is linear but time-varying: the factor $n$ changes with time. The moving average is **linear and time-invariant** — convolution with a fixed impulse response $h=[1/3,1/3,1/3]$ — hence LTI.

## Task 3.4 — Finding a hidden delay with cross-correlation

In [ ]:
rng = np.random.default_rng(1)
fs = 1000
xt = rng.standard_normal(500)
true_delay = 37
for sd in [0.5, 1, 2, 4]:
    y = np.concatenate([np.zeros(true_delay), 0.6*xt])[:500] + sd*rng.standard_normal(500)
    r = signal.correlate(y, xt, mode='full')
    lags = signal.correlation_lags(len(y), len(xt), mode='full')
    est = lags[np.argmax(r)]
    print(f'noise sd = {sd:4.1f}: estimated delay = {est:4d} samples '
          f'({est/fs*1000:6.1f} ms, {est/fs*343:5.2f} m)')
    if sd == 0.5:
        plt.figure(figsize=(9, 3))
        plt.plot(lags, r); plt.xlim(-200, 200)
        plt.title(f'Task 3.4: cross-correlation, noise sd = {sd}')
        plt.xlabel('lag (samples)'); plt.ylabel('correlation'); plt.grid(alpha=0.3)
        plt.tight_layout(); plt.show()

With this RNG seed the estimate stays at 37 samples up to noise sd = 4; with a fresh noise draw at sd = 4 it already gives 87 samples, and at sd ≥ 6 it fails completely (the correlation peak is no longer dominant). Distance from delay: $d = 343\,\mathrm{m/s} \times (37/1000)\,\mathrm{s} \approx 12.7\,\mathrm{m}$.

## Task 3.5 — Difference equation

$y(n)-\tfrac34 y(n-1)+\tfrac18 y(n-2)=x(n)+x(n-1)$

In [ ]:
N = 10
imp = np.zeros(N); imp[0] = 1
h_rec = np.zeros(N)
for n in range(N):
    x0 = imp[n]
    x1 = imp[n-1] if n >= 1 else 0
    y1 = h_rec[n-1] if n >= 1 else 0
    y2 = h_rec[n-2] if n >= 2 else 0
    h_rec[n] = x0 + x1 + 0.75*y1 - 0.125*y2
h_lfilter = signal.lfilter([1, 1], [1, -0.75, 0.125], imp)
n = np.arange(N)
h_closed = 6*(0.5**n) - 5*(0.25**n)
for label, h in [('recursion', h_rec), ('lfilter', h_lfilter), ('closed form', h_closed)]:
    print(f'{label:12s}', np.round(h, 5))
plt.stem(n, h_rec, basefmt=' ', label='recursion')
plt.plot(n, h_closed, 'ro--', label='closed form')
plt.title('Task 3.5: impulse response'); plt.xlabel('n (samples)')
plt.ylabel('h(n)'); plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

step = signal.lfilter([1, 1], [1, -0.75, 0.125], np.ones(80))
H1 = np.sum([1, 1]) / np.sum([1, -0.75, 0.125])
print(f'step final value = {step[-1]:.5f}, H(1) = {H1:.5f}')

**Checkpoint 3:** show the Task 3.3 output and the three matching impulse responses from Task 3.5 (they agree to five decimal places).

## Analysis questions

**1.** `np.convolve` runs compiled, vectorised C/FFT-based code, while the Python double loop pays interpreter overhead for every index. Direct convolution needs $L\times M$ multiplications (for $L=M=2000$, four million); FFT convolution needs about $O(N\log N)$ operations.

**2.** Only the moving average is LTI. $x^2$ is nonlinear; $n\,x(n)$ is time-varying. This is predictable from the definitions: an LTI system must be a fixed convolution and its rule must not depend explicitly on $n$.

**3.** Two Tanzanian applications: (i) locating leaks in water pipes by correlating acoustic/vibration sensors at two points (time delay → position); (ii) radar/sonar at airports or ports for estimating target range from echo delay. GPS pseudo-random code correlation is another example.

**4.** **IIR**, because the equation is recursive (non-zero denominator coefficients $a_1=-0.75$, $a_2=0.125$). It is **stable**: poles are at $z=0.5$ and $z=0.25$, both inside the unit circle, and $h(n)=6(0.5)^n-5(0.25)^n$ decays to zero and is absolutely summable.

## Deliverables checklist

- [ ] Tasks 3.1–3.5 complete; code runs top to bottom
- [ ] Timing comparison, plots and checkpoint 3 evidence included
- [ ] Pre-lab and analysis answers included
- [ ] AI-use statement included
- [ ] `COU07506_Week03_<RegNo>.ipynb` uploaded to DIT-eLMS

## AI-use statement

Tool used: OpenCode (Big Pickle) chat assistant.

Tasks it helped with: explaining the lab brief, suggesting the Python code structure and checking the numerical answers (alias frequencies, SQNR, convolution, impulse response).

How the results were checked: the 7 kHz @ 10 kHz case was checked by hand against `|F - k fs|`; the convolution was checked against `np.convolve`; the 6 dB-per-bit rule was checked against `6.02B + 1.76`; the difference-equation impulse response was checked against the closed form `6(0.5)^n - 5(0.25)^n`; plots were inspected for correct labels and expected shapes. All submitted code was reviewed line by line.